###  Load Data & Initialize Master Data

In [0]:
# 1. Read the raw table exactly as it is in the catalog
raw_df = spark.table("default.superstore")

# 2. Select the columns using their exact space-containing names
selected_df = raw_df.select(
    "`Customer ID`", "`Customer Name`", "Segment", "Country", "City", "State"
).distinct()

# 3. Rename the space-containing columns to use clean underscores
clean_df = selected_df \
    .withColumnRenamed("Customer ID", "Customer_ID") \
    .withColumnRenamed("Customer Name", "Customer_Name")

# 4. Save the clean schema as your persistent assignment Delta table
clean_df.write.format("delta").mode("overwrite").saveAsTable("delta_customer_master")

print("Initial Master Delta Table successfully created with clean column names!")
display(spark.table("delta_customer_master").limit(5))

Initial Master Delta Table successfully created with clean column names!


Customer_ID,Customer_Name,Segment,Country,City,State
PK-19075,Pete Kriz,Consumer,United States,Madison,Wisconsin
PO-18865,Patrick O'Donnell,Consumer,United States,Westland,Michigan
JS-15685,Jim Sink,Corporate,United States,Los Angeles,California
GZ-14470,Gary Zandusky,Consumer,United States,Rochester,Minnesota
JM-15250,Janet Martin,Consumer,United States,Charlotte,North Carolina


### Basic Cleaning (Handle Nulls & Duplicates)

In [0]:
# Read from our newly created Delta Table
master_df = spark.table("delta_customer_master")

# Drop any duplicate rows for the same Customer_ID
cleaned_df = master_df.dropDuplicates(["Customer_ID"])

# Standardize missing strings with a fallback value 'Unknown'
cleaned_df = cleaned_df.na.fill("Unknown", subset=["Segment", "City", "State"])

# Overwrite the Delta table with the verified clean data
cleaned_df.write.format("delta").mode("overwrite").saveAsTable("delta_customer_master")

print(f"Cleaned Master Row Count: {cleaned_df.count()}")

Cleaned Master Row Count: 793


### Create Incremental / New Data Simulation

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

# Define clean structural layout matching our master data column names
schema = StructType([
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True)
])

# Simulating incoming incremental batch data
incremental_data = [
    ("CG-12520", "Claire Gute", "Corporate", "United States", "Miami", "Florida"),  # City updated to Miami
    ("DV-13045", "Darrin Van Huff", "Corporate", "United States", "Austin", "Texas"), # City updated to Austin
    ("NEW-99991", "Alex Mercer", "Consumer", "United States", "New York", "New York"), # Brand new customer
    ("NEW-99992", "Sarah Connor", "Home Office", "United States", "Los Angeles", "California") # Brand new customer
]

incremental_df = spark.createDataFrame(incremental_data, schema)
incremental_df.write.format("delta").mode("overwrite").saveAsTable("customer_incremental_batch")

print("Incremental update batch dataset prepared:")
display(incremental_df)

Incremental update batch dataset prepared:


Customer_ID,Customer_Name,Segment,Country,City,State
CG-12520,Claire Gute,Corporate,United States,Miami,Florida
DV-13045,Darrin Van Huff,Corporate,United States,Austin,Texas
NEW-99991,Alex Mercer,Consumer,United States,New York,New York
NEW-99992,Sarah Connor,Home Office,United States,Los Angeles,California


### Apply Delta Lake MERGE Operation (SCD Type 1)

In [0]:
from delta.tables import DeltaTable

# Load the target master delta table object
target_delta_table = DeltaTable.forName(spark, "delta_customer_master")
source_updates_df = spark.table("customer_incremental_batch")

# Execute atomic MERGE upsert operation
target_delta_table.alias("target") \
  .merge(
    source_updates_df.alias("updates"),
    "target.Customer_ID = updates.Customer_ID"
  ) \
  .whenMatchedUpdate(set = {
    "Customer_Name": "updates.Customer_Name",
    "Segment": "updates.Segment",
    "Country": "updates.Country",
    "City": "updates.City",
    "State": "updates.State"
  }) \
  .whenNotMatchedInsert(values = {
    "Customer_ID": "updates.Customer_ID",
    "Customer_Name": "updates.Customer_Name",
    "Segment": "updates.Segment",
    "Country": "updates.Country",
    "City": "updates.City",
    "State": "updates.State"
  }) \
  .execute()

print("Delta Merge (SCD Type 1 Upsert) executed successfully.")

Delta Merge (SCD Type 1 Upsert) executed successfully.


### Pipeline Data Validation

In [0]:
final_df = spark.table("delta_customer_master")

# Verify targeted records reflect changes properly
validated_rows = final_df.filter(
    (final_df["Customer_ID"] == "CG-12520") | 
    (final_df["Customer_ID"] == "NEW-99991")
)
print("Validation Check - Updated & Inserted Rows:")
display(validated_rows)

# Confirm constraints remain intact (no duplicate primary keys)
duplicate_count = final_df.groupBy("Customer_ID").count().filter("count > 1").count()
print(f"Validation Check - Total Duplicates Found: {duplicate_count}")

Validation Check - Updated & Inserted Rows:


Customer_ID,Customer_Name,Segment,Country,City,State
CG-12520,Claire Gute,Corporate,United States,Miami,Florida
NEW-99991,Alex Mercer,Consumer,United States,New York,New York


Validation Check - Total Duplicates Found: 0


### Final Table Audit & History Summary

In [0]:
# Display transaction history logs
history_df = target_delta_table.history()
print("Delta Table Transaction Audit Log:")
display(history_df.select("version", "timestamp", "operation", "operationMetrics").limit(5))

print(f"Final Total Record Count: {final_df.count()}")

Delta Table Transaction Audit Log:


version,timestamp,operation,operationMetrics
2,2026-07-05T09:16:02.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2242, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 5548, materializeSourceTimeMs -> 11, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2976, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2442)"
1,2026-07-05T09:14:30.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 37203, numDeletionVectorsRemoved -> 0, numOutputRows -> 793, numOutputBytes -> 15572)"
0,2026-07-05T09:13:40.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 4688, numOutputBytes -> 37203)"


Final Total Record Count: 795
